## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import powerlaw
import os
import sys
from eda_toolkit import ensure_directory, generate_table1

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

sys.path.append("../")

## Paths

In [ ]:
data_path = "../data/raw"

save_dir_png = "../images/png_images"
save_dir_svg = "../images/svg_images"

os.makedirs(save_dir_png, exist_ok=True)
os.makedirs(save_dir_svg, exist_ok=True)

In [ ]:
base_path = os.path.join(os.pardir)

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

In [ ]:
df = pd.read_csv(os.path.join(data_path, "ACLED Data_2026-01-02.csv")).set_index(
    "event_id_cnty"
)

In [ ]:
df.head()

In [ ]:
df["disorder_type"].value_counts()

In [ ]:
df["fatality_yes_no"] = np.where(df["fatalities"] > 0, "Yes", "No")

In [ ]:
df["fatality_yes_no"]

In [ ]:
df["fatality_yes_no"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

counts_dict = {"No (zero fatalities)": 136375, "Yes (\u22651 fatality)": 21724}
labels = list(counts_dict.keys())
counts = list(counts_dict.values())
total = sum(counts)
pcts = [c / total * 100 for c in counts]

fig, ax = plt.subplots(figsize=(8, 4.0))

x = np.array([0, 1])
bars = ax.bar(x, counts,
              width=1.0,
              color=["#4a6fa5", "#c25450"],
              edgecolor="white", linewidth=1.2)

for bar, c, p in zip(bars, counts, pcts):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2,
            h + total * 0.012,
            f"{c:,}\n({p:.1f}%)",
            ha="center", va="bottom",
            fontsize=11, color="#222")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlim(-0.5, 1.5)

ax.set_ylabel("Event count", fontsize=12)
ax.set_title("Class distribution of fatality occurrence (N = 158,099)",
             fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(0, max(counts) * 1.18)

yticks = [0, 25000, 50000, 75000, 100000, 125000]
ax.set_yticks(yticks)
ax.set_yticklabels([f"{y // 1000:d}K" if y else "0" for y in yticks])
ax.tick_params(axis="both", labelsize=11)

plt.tight_layout()

plt.savefig(f"{save_dir_png}/class_distribution.png", dpi=400, bbox_inches="tight")
plt.savefig(f"{save_dir_svg}/class_distribution.svg", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd

# Adjust column names to match your dataframe
sub = (
    df.groupby("sub_event_type")
      .agg(event_count=("fatalities", "size"),
           fatal_events=("fatalities", lambda s: (s > 0).sum()),
           total_fatalities=("fatalities", "sum"),
           max_fatalities=("fatalities", "max"))
      .reset_index()
)

sub["mean_fatalities"] = sub["total_fatalities"] / sub["event_count"]
sub["pct_of_events"] = sub["event_count"] / sub["event_count"].sum() * 100

# The 7 sub-types in the "Other" footnote of Table 1, sorted ascending
rare_types = [
    "Suicide bomb",
    "Chemical weapon",
    "Violent demonstration",
    "Mob violence",
    "Headquarters or base established",
    "Excessive force against protesters",
    "Protest with intervention",
]

rare = (
    sub[sub["sub_event_type"].isin(rare_types)]
       .sort_values("event_count")
       .reset_index(drop=True)
)

# Format for display
disp = rare[["sub_event_type", "event_count", "pct_of_events",
             "fatal_events", "total_fatalities", "max_fatalities", "mean_fatalities"]].copy()
disp["pct_of_events"] = disp["pct_of_events"].map(lambda x: f"{x:.3f}%")
disp["mean_fatalities"] = disp["mean_fatalities"].map(lambda x: f"{x:.2f}")
disp.columns = ["Sub-event type", "Events", "% of total", "Fatal events",
                "Total fatalities", "Max fatalities", "Mean fatalities/event"]

print(disp.to_string(index=False))

In [ ]:
df.select_dtypes(include="number").columns

In [ ]:
df.head()

In [ ]:
################################################################################
# 1. Load data
################################################################################

# df = pd.read_parquet("data/acled_ukraine_full.parquet")  # adjust path
y = df["fatalities"].values.astype(float)
n = len(y)
total_fatalities = y.sum()

################################################################################
# 2. Empirical concentration (panel a)
################################################################################
y_sorted = np.sort(y)[::-1]
cum_fat = np.cumsum(y_sorted) / total_fatalities
cum_evt = np.arange(1, n + 1) / n
auc_perfect = np.trapezoid(cum_fat, cum_evt)

# Concentration thresholds
thresholds = [0.01, 0.05, 0.10]
threshold_pts = []
for t in thresholds:
    k = max(1, int(np.ceil(t * n)))
    threshold_pts.append((t, cum_fat[k - 1]))

################################################################################
# 3. Power-law fit on positive tail (panel b)
################################################################################

y_pos = y[y > 0].astype(int)
fit = powerlaw.Fit(y_pos, discrete=True, verbose=False)
alpha = fit.power_law.alpha
xmin = fit.power_law.xmin

xs_emp = np.sort(np.unique(y_pos))
ccdf_emp = np.array([(y_pos >= x).mean() for x in xs_emp])

xs_fit = np.logspace(np.log10(xmin), np.log10(y_pos.max()), 80)
ccdf_at_xmin = (y_pos >= xmin).mean()
ys_fit = ccdf_at_xmin * (xs_fit / xmin) ** -(alpha - 1)

################################################################################
# 4. Figure
################################################################################

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

# Panel (a): empirical concentration curve
ax = axes[0]   
ax.fill_between(cum_evt, cum_evt, cum_fat, color="#1d3557", alpha=0.07)
ax.plot(cum_evt, cum_fat, color="#1d3557", lw=1.5,
        label=f"Empirical (AUC = {auc_perfect:.3f})")
ax.plot([0, 1], [0, 1], color="gray", ls="--", lw=0.8,
        label="Random ranking (AUC = 0.500)")

# Threshold annotations: place to the right of each marker

annot_offsets = {
    0.01: ( 0.08, -0.18),
    0.05: ( 0.14, -0.18),
    0.10: ( 0.22, -0.14),
}
for (t, val) in threshold_pts:
    dx, dy = annot_offsets[t]
    ax.plot(t, val, "o", color="#c25450", ms=4, zorder=5)
    ax.annotate(f"{val * 100:.0f}% of fatalities\nin top {int(t * 100)}% of events",
                xy=(t, val),
                xytext=(t + dx, val + dy),
                fontsize=8.5, color="#333",
                arrowprops=dict(arrowstyle="-", color="#888", lw=0.5))

ax.set_xlabel("Fraction of events (ranked by actual fatalities)")
ax.set_ylabel("Cumulative fraction of fatalities")
ax.set_title("(a) Empirical concentration of fatalities")
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.04)

# Panel (b): tail CCDF with power-law overlay
ax = axes[1]
ax.loglog(xs_emp, ccdf_emp, "o", ms=3, color="#1d3557", alpha=0.6,
          label="Empirical CCDF")
ax.loglog(xs_fit, ys_fit, color="#c25450", lw=1.5,
          label=fr"Power-law fit ($\alpha$ = {alpha:.2f}, $x_{{\min}}$ = {int(xmin)})")

# Mark x_min as a faint vertical guide
ax.axvline(xmin, color="#888", ls=":", lw=0.6, alpha=0.6)

ax.set_xlabel("Fatalities per event")
ax.set_ylabel(r"$P(X \geq x)$")
ax.set_title("(b) Tail distribution and power-law fit")
ax.legend(loc="lower left", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{save_dir_png}/empirical_concentration.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{save_dir_svg}/empirical_concentration.svg", bbox_inches="tight")
plt.show()

################################################################################
# 5. Console summary (for paper text and table)
################################################################################

print(f"n = {n:,}")
print(f"Perfect-information capture AUC = {auc_perfect:.4f}")
print(f"alpha = {alpha:.3f},  x_min = {int(xmin)},  n above x_min = {(y_pos >= xmin).sum():,}")
print()
for (t, val) in threshold_pts:
    k = max(1, int(np.ceil(t * n)))
    print(f"Top {int(t * 100):>2}% ({k:>6,} events): {val * 100:5.1f}% of fatalities")

## Table 1

In [ ]:
df_table1 = df[
    [
        "sub_event_type",
        "fatality_yes_no",
    ]
]

In [ ]:
df_table1["sub_event_type"].value_counts()

In [ ]:
df_table1

In [ ]:
table_1 = generate_table1(
    df,
    value_counts=True,
    groupby_col="fatality_yes_no",
    drop_variables="fatality_yes_no",
    categorical_cols=["sub_event_type"],
    continuous_cols=["time_precision", "geo_precision", "latitude", "longitude"],
)

In [ ]:
table_1 = table_1.drop(
    columns=[
        "Type",
        "Mode",
        "Mean",
        "SD",
        "Median",
        "Min",
        "Max",
        "Missing (n)",
        "Missing (%)",
    ]
)

In [ ]:
df["civilian_targeting"].value_counts()

In [ ]:
table_1

## Fatality Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

zero_pct = (df["fatalities"] == 0).mean() * 100
log_all = np.log1p(df["fatalities"])
log_fatal = np.log1p(df[df["fatalities"] > 0]["fatalities"])

### Panel 1: log-transformed, ALL events
ax1 = fig.add_subplot(gs[0])

ax1.hist(
    log_all,
    bins=50,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
    density=True,
    alpha=0.75,
)

ax1.axvline(
    log_all.mean(),
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"Mean = {log_all.mean():.2f}",
)
ax1.axvline(
    df["fatalities"].median(),
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"Median = {int(df['fatalities'].median())}",
)


ax1.set_xlabel("log(fatalities + 1)", fontsize=13)
ax1.set_ylabel("Density", fontsize=13)
ax1.set_title(f"All events (n = {len(df):,})", fontsize=12)
ax1.legend(fontsize=12, framealpha=0.9)
ax1.tick_params(labelsize=11)

## Panel 2: log-transformed, FATAL events only
ax2 = fig.add_subplot(gs[1])

ax2.hist(
    log_fatal,
    bins=40,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
    density=True,
    alpha=0.6,
)

kde = gaussian_kde(log_fatal)
x_range = np.linspace(log_fatal.min(), log_fatal.max(), 300)
ax2.plot(x_range, kde(x_range), color="red", linewidth=2.5, label="KDE")

mu, sigma = log_fatal.mean(), log_fatal.std()
ax2.axvline(mu, color="blue", linestyle="--", linewidth=2, label=f"Mean = {mu:.2f}")
ax2.axvline(
    log_fatal.median(),
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"Median = {log_fatal.median():.2f}",
)
ax2.axvspan(mu - sigma, mu + sigma, alpha=0.08, color="purple", label="±1 SD")
ax2.axvspan(mu - 2 * sigma, mu + 2 * sigma, alpha=0.05, color="green", label="±2 SD")

ax2.set_xlim(left=0)
ax2.set_xlabel("log(fatalities + 1)", fontsize=13)
ax2.set_ylabel("Density", fontsize=13)
ax2.set_title(f"Fatal events only (n = {len(log_fatal):,})", fontsize=12)
ax2.legend(fontsize=11, framealpha=0.9)
ax2.tick_params(labelsize=11)

# fig.suptitle(
#     f"Fatality distribution — {zero_pct:.1f}% of events report zero fatalities",
#     fontsize=14,
#     fontweight="bold",
#     y=1.02,
# )

plt.savefig(
    os.path.join(image_path_png, "fatality_distribution.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    os.path.join(image_path_svg, "fatality_distribution.svg"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.5)

### Panel 1: Mean fatalities by interaction type
ax1 = fig.add_subplot(gs[0])

interaction_means = (
    df.groupby("interaction")["fatalities"].mean().sort_values(ascending=True)
)

bars1 = ax1.barh(
    interaction_means.index.astype(str),
    interaction_means.values,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
    height=0.65,
)

# Value labels
for bar, val in zip(bars1, interaction_means.values):
    ax1.text(
        val + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=10,
        color="dimgray",
    )

ax1.set_xlabel("Mean fatalities per event", fontsize=12)
ax1.set_ylabel("Interaction code", fontsize=12)
ax1.set_title("Mean fatalities by interaction type", fontsize=13, fontweight="bold")
ax1.tick_params(labelsize=10)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.set_xlim(0, interaction_means.max() * 1.2)

### Panel 2: Mean fatalities by sub_event_type
ax2 = fig.add_subplot(gs[1])

subtype_means = (
    df.groupby("sub_event_type")["fatalities"].mean().sort_values(ascending=True)
)

## Color-code: top 3 in coral, rest in steelblue
colors = [
    "#d85a30" if v >= subtype_means.nlargest(3).min() else "steelblue"
    for v in subtype_means.values
]

bars2 = ax2.barh(
    subtype_means.index,
    subtype_means.values,
    color=colors,
    edgecolor="white",
    linewidth=0.5,
    height=0.65,
)

## Value labels
for bar, val in zip(bars2, subtype_means.values):
    ax2.text(
        val + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=9,
        color="dimgray",
    )

ax2.set_xlabel("Mean fatalities per event", fontsize=12)
ax2.set_ylabel("Sub-event type", fontsize=12)
ax2.set_title("Mean fatalities by sub-event type", fontsize=13, fontweight="bold")
ax2.tick_params(labelsize=9)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_xlim(0, subtype_means.max() * 1.2)

# Top 3 legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#d85a30", label="Top 3 by mean fatalities"),
    Patch(facecolor="steelblue", label="All others"),
]
ax2.legend(handles=legend_elements, fontsize=10, framealpha=0.9, loc="lower right")

# fig.suptitle(
#     "Mean fatalities per event by interaction type and sub-event type",
#     fontsize=14,
#     fontweight="bold",
#     y=1.02,
# )

plt.savefig(
    os.path.join(image_path_png, "mean_fatalities_by_feature.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    os.path.join(image_path_svg, "mean_fatalities_by_feature.svg"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()